# 01 环境安全与一键体检

**用途：** 检查虚拟环境、依赖、密钥安全、67条测试和两套30条业务回归。

> 使用方式：按顺序运行。出现 `PASS` 才代表本节验收成功；断言失败时先阅读紧邻的“失败定位”。默认不调用真实模型、不写生产数据库。

In [1]:
from pathlib import Path
import importlib.util
import json
import os
import sys
import tempfile

cwd = Path.cwd().resolve()
DAY1_ROOT = None
PROJECT2_ROOT = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / "project2" / "agent_graph.py").exists():
        DAY1_ROOT = candidate
        PROJECT2_ROOT = candidate / "project2"
        break
    if (candidate / "agent_graph.py").exists() and (candidate / "tests").exists():
        PROJECT2_ROOT = candidate
        DAY1_ROOT = candidate.parent
        break
assert DAY1_ROOT is not None and PROJECT2_ROOT is not None, "找不到 day1/project2 项目根目录"
NOTEBOOK_ROOT = PROJECT2_ROOT / "notebooks"
for path in [str(DAY1_ROOT), str(PROJECT2_ROOT), str(NOTEBOOK_ROOT)]:
    if path not in sys.path:
        sys.path.insert(0, path)

from notebook_utils import (
    check,
    check_equal,
    file_inventory,
    load_jsonl,
    masked_environment,
    run_command,
    run_unittest,
    show_markdown,
    show_table,
    source_excerpt,
)

RUN_LIVE_MODEL_TESTS = os.getenv("RUN_LIVE_MODEL_TESTS", "0") == "1"
print(f"Python: {sys.executable}")
print(f"DAY1_ROOT: {DAY1_ROOT}")
print(f"PROJECT2_ROOT: {PROJECT2_ROOT}")
print(f"RUN_LIVE_MODEL_TESTS: {RUN_LIVE_MODEL_TESTS}")

Python: D:\new things\项目1\day1\.venv\Scripts\python.exe
DAY1_ROOT: D:\new things\项目1\day1
PROJECT2_ROOT: D:\new things\项目1\day1\project2
RUN_LIVE_MODEL_TESTS: False


## 1. 环境和密钥

Notebook只展示“是否配置”，永远不打印Key。`.env`只在本机使用，Streamlit Cloud使用Secrets。

**失败定位：**
- 导入失败：确认内核是 `Python (.venv 项目1 Agent)`。
- 测试数量不是67：代码或测试文件没有同步。
- LIVE模型失败：先区分鉴权、限流、超时和结构化输出错误。

In [2]:
package_names = [
    "streamlit", "langchain", "langgraph", "chromadb",
    "pydantic", "nbformat", "nbclient", "ipykernel",
]
package_rows = [
    {"依赖": name, "可导入": bool(importlib.util.find_spec(name))}
    for name in package_names
]
show_table(package_rows)
check("Notebook及Agent依赖可导入", all(row["可导入"] for row in package_rows))

key_rows = masked_environment(["DEEPSEEK_API_KEY", "ZHIPU_API_KEY"])
show_table(key_rows)
check("密钥没有明文显示", all(row["显示值"] in {"", "***"} for row in key_rows))

,依赖,可导入
0,streamlit,True
1,langchain,True
2,langgraph,True
3,chromadb,True
4,pydantic,True
5,nbformat,True
6,nbclient,True
7,ipykernel,True


[PASS] Notebook及Agent依赖可导入


,变量,状态,显示值
0,DEEPSEEK_API_KEY,未配置,
1,ZHIPU_API_KEY,未配置,


[PASS] 密钥没有明文显示


{'检查项': '密钥没有明文显示', '状态': 'PASS', '说明': ''}

In [3]:
all_tests = run_command(
    [sys.executable, "-m", "unittest", "discover", "-s", "tests", "-p", "test_*.py", "-q"],
    cwd=PROJECT2_ROOT,
    timeout=240,
)
check("67条测试全部通过", "Ran 67 tests" in all_tests.output and "OK" in all_tests.output)

$ D:\new things\项目1\day1\.venv\Scripts\python.exe -m unittest discover -s tests -p test_*.py -q
----------------------------------------------------------------------
Ran 67 tests in 2.517s

OK
[PASS] 67条测试全部通过


{'检查项': '67条测试全部通过', '状态': 'PASS', '说明': ''}

In [4]:
workflow_eval = run_command(
    [sys.executable, "tests/evaluate_agent.py", "--mode", "workflow"],
    cwd=PROJECT2_ROOT,
)
graph_eval = run_command(
    [sys.executable, "tests/evaluate_agent.py", "--mode", "graph"],
    cwd=PROJECT2_ROOT,
)
check("workflow 30/30", "Passed: 30" in workflow_eval.output and "100.0%" in workflow_eval.output)
check("LangGraph 30/30", "Passed: 30" in graph_eval.output and "100.0%" in graph_eval.output)

$ D:\new things\项目1\day1\.venv\Scripts\python.exe tests/evaluate_agent.py --mode workflow
Total: 30
Mode: workflow
Passed: 30
Pass rate: 100.0%
CSV report: D:\new things\��Ŀ1\day1\project2\reports\agent_evaluation_workflow.csv
Summary report: D:\new things\��Ŀ1\day1\project2\reports\agent_evaluation_summary_workflow.md


$ D:\new things\项目1\day1\.venv\Scripts\python.exe tests/evaluate_agent.py --mode graph
Total: 30
Mode: graph
Passed: 30
Pass rate: 100.0%
CSV report: D:\new things\��Ŀ1\day1\project2\reports\agent_evaluation_graph.csv
Summary report: D:\new things\��Ŀ1\day1\project2\reports\agent_evaluation_summary_graph.md
[PASS] workflow 30/30
[PASS] LangGraph 30/30


{'检查项': 'LangGraph 30/30', '状态': 'PASS', '说明': ''}

## 面试官会问

- 为什么单元测试通过不等于线上可用？
- 如何区分离线测试、真实API冒烟、业务准确率和生产SLA？
- API Key怎么管理？日志为什么不能保存Prompt和客户隐私？
- 你怎样保证Notebook不会污染正式数据库？

### 参考答案

1. **为什么测试通过不等于线上可用？** 单元测试只覆盖已知输入和受控依赖，线上还会出现网络抖动、Provider限流、脏数据、并发、权限、部署重启和分布漂移，所以还需要真实API冒烟、线上采样评测、SLA、告警和回滚。
2. **四类验证如何区分？** 离线测试验证确定性逻辑；真实API冒烟验证接口、鉴权和Schema；业务准确率必须对人工gold计算检索/字段/拒识指标；生产SLA再看可用性、P95延迟、错误率、成本和恢复时间。
3. **API Key和隐私怎么管理？** Key只放`.env`或部署Secret，不提交Git、不写Notebook输出；日志只保存模型、状态、Token估算、错误类别等必要元数据并执行凭据脱敏。原始Prompt可能包含电话、订单和客户需求，默认不进入模型日志。
4. **Notebook怎样不污染正式数据库？** checkpoint、会话、记忆和服务单示例都通过依赖注入指向`TemporaryDirectory`下的SQLite，单元结束后关闭连接并清理；只有显式运行正式应用才使用`project2/logs/`。

**代码落点：** `notebook_utils.py`、`agent_harness.py::sanitize_error_message`、各Notebook中的`TemporaryDirectory`和仓库构造代码。

**成功标准：** 67/67、workflow 30/30、LangGraph 30/30。图片40/40预跑不在这里算准确率。